# Эксперимент 01 — MLE → loss → модель-артефакт → доверительные интервалы

**Тип:** Уровень 2 (расширенный). Стартует от идей главы 5 UDL, доводит до артефакта и CI.

**Гипотеза на бумаге (заполни ДО запуска):**
- Предположение о данных: шум гауссов, `y = a + b·x + ε`, `ε ~ N(0, σ²)`.
- Из MLE следует: оптимальный loss = MSE (вывод — в `labkit.theory.gaussian_nll`).
- **Предсказание:** оценки коэффициентов близки к истинным `[a, b]`; 95% CI накроют истину.
- **Предсказание:** замена на лапласовское предположение (MAE) даст похожие, но не идентичные оценки.

> Правило лаборатории: предсказание записывается ДО эксперимента. Иначе это подгонка, а не проверка.


## 1. Сетап (тонко — вся логика в labkit)

In [1]:
import numpy as np
from labkit.engine import set_seed, get_device, RunLogger
from labkit.theory import gaussian_nll, laplace_nll, confidence_intervals

set_seed(0)
device = get_device()
log = RunLogger("exp01_mle_loss")

torch не установлен — только теоретическая часть


## 2. Данные (известная истина — чтобы проверять предсказание)

In [2]:
TRUE = np.array([2.0, -1.0])   # [a, b] — что модель должна восстановить
SIGMA = 0.1
x = np.linspace(0, 1, 80)
y = TRUE[0] + TRUE[1] * x + np.random.normal(0, SIGMA, x.size)

## 3. Оценка через MLE
MSE-решение = MLE при гауссовом шуме. Здесь — замкнутая форма (МНК).

In [3]:
A = np.vstack([np.ones_like(x), x]).T
theta_hat = np.linalg.lstsq(A, y, rcond=None)[0]
print("оценка [a, b]:", np.round(theta_hat, 3), "| истина:", TRUE)

оценка [a, b]: [ 2.055 -1.117] | истина: [ 2. -1.]


## 4. Доверительные интервалы (твоя авторская добавка поверх книги)
Гессиан NLL в минимуме = информация Фишера → обращаем → SE → CI.

In [4]:
def nll(theta):
    return gaussian_nll(y, theta[0] + theta[1] * x, sigma=SIGMA)

lo, hi, se = confidence_intervals(nll, theta_hat, level=0.95)
for i, name in enumerate(["a", "b"]):
    covered = lo[i] <= TRUE[i] <= hi[i]
    print(f"{name}: {theta_hat[i]:.3f}  95% CI [{lo[i]:.3f}, {hi[i]:.3f}]  "
          f"истина накрыта: {covered}")
    log.log(param=name, est=float(theta_hat[i]), ci_lo=float(lo[i]),
            ci_hi=float(hi[i]), covered=bool(covered))

a: 2.055  95% CI [2.012, 2.099]  истина накрыта: False
b: -1.117  95% CI [-1.192, -1.042]  истина накрыта: False


## 5. Контраст: другое предположение о шуме → другой loss
Если бы шум был лапласовский, MLE → MAE. Сравним loss при истинных параметрах.

In [5]:
g = gaussian_nll(y, A @ theta_hat, sigma=SIGMA)
l = laplace_nll(y, A @ theta_hat, b=SIGMA)
print(f"NLL гауссов (→MSE): {g:.2f}")
print(f"NLL лаплас  (→MAE): {l:.2f}")
print("Вывод: loss — следствие предположения о данных, а не выбор из меню.")

NLL гауссов (→MSE): -74.88
NLL лаплас  (→MAE): -69.79
Вывод: loss — следствие предположения о данных, а не выбор из меню.


## 6. Сверка предсказания с результатом (заполни ПОСЛЕ запуска)

| Предсказание | Результат | Сошлось? |
|---|---|---|
| оценки близки к [2.0, -1.0] | … | … |
| 95% CI накрывают истину | … | … |
| MAE-loss ≠ MSE-loss | … | … |

**Что узнал / где теория ошиблась:**
_…твои заметки сюда…_


In [6]:
log.close()
print("эксперимент закрыт, метрики в logs/exp01_mle_loss.jsonl")

эксперимент закрыт, метрики в logs/exp01_mle_loss.jsonl
